<a href="https://colab.research.google.com/github/tashir0605/Cocepts-And-Practice/blob/main/Capstone%20Project/Copy_of_LightGBM_Logging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mlflow==2.12.2 boto3 awscli optuna imbalanced-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
from google.colab import userdata

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = userdata.get("AWS_DEFAULT_REGION")

In [ ]:
import mlflow


In [ ]:
mlflow.set_tracking_uri("http://13.63.238.146:5000")

In [ ]:
mlflow.set_experiment("Exp 5 - Model")

<Experiment: artifact_location='s3://tashir-mlflow-bucket/7', creation_time=1780043875633, experiment_id='7', last_update_time=1780043875633, lifecycle_stage='active', name='Exp 5 - Model', tags={}>

In [ ]:
import pandas as pd
df=pd.read_csv("/content/reddit_preprocessing.csv").dropna(subset=['clean_comment'])
df.shape
df.head()

,clean_comment,category,word_count,num_stop_words,num_chars,num_punctuation_chars
0,family mormon never tried explain still stare ...,1,39,13,259,0
1,buddhism much lot compatible christianity espe...,1,196,59,1268,0
2,seriously say thing first get complex explain ...,-1,86,40,459,0
3,learned want teach different focus goal not wr...,0,29,15,167,0
4,benefit may want read living buddha living chr...,1,112,45,690,0


In [ ]:
import optuna
import mlflow
import mlflow.sklearn

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.svm import SVC

# Step 1: Remap the class labels from [-1, 0, 1] to
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# --- SPLIT ORIGINAL RAW TEXT DATA FIRST (Prevents Leakage) ---
X_raw = df['clean_comment']
y_raw = df['category']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)


# Function to log results in MLflow
def log_mlflow(model_name, model, vectorizer, X_train_raw, X_test_raw, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_ClassWeights_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        # 1. Fit/Transform vectorizer ONLY on training set text
        X_train_vec = vectorizer.fit_transform(X_train_raw)
        X_test_vec = vectorizer.transform(X_test_raw)

        # 2. Compute class balance weights ONLY for the training records
        # This penalizes the model more when it misclassifies minority classes
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

        # 3. Train model using sample weights
        model.fit(X_train_vec, y_train, sample_weight=sample_weights)
        y_pred = model.predict(X_test_vec)

        # Log metrics
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log nested classification report metrics
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log both artifacts
        mlflow.sklearn.log_model(model, f"{model_name}_model")
        mlflow.sklearn.log_model(vectorizer, f"{model_name}_vectorizer")


# Step 6: Optuna objective function with Class Weights
def objective_svm(trial):
    C = trial.suggest_float('C', 1e-3, 1e2, log=True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])

    # 1. Isolate TF-IDF fitting to training split
    vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=1000)
    X_train_vec = vectorizer.fit_transform(X_train_raw)
    X_test_vec = vectorizer.transform(X_test_raw)

    # 2. Compute dynamic weights inside optimization step
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    # 3. Initialize and evaluate model
    model = SVC(
        C=C,
        kernel=kernel,
        random_state=42
    )

    model.fit(X_train_vec, y_train, sample_weight=sample_weights)
    y_pred = model.predict(X_test_vec)
    return accuracy_score(y_test, y_pred)


# Step 7: Run Optuna for SVM, log the best model setup
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_svm, n_trials=30)

    best_params = study.best_params
    best_model = SVC(
        C=best_params['C'],
        kernel=best_params['kernel'],
        random_state=42
    )

    final_vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=1000)

    # Log the baseline objects cleanly
    log_mlflow("SVM", best_model, final_vectorizer, X_train_raw, X_test_raw, y_train, y_test)

# Run the experiment for SVM
run_optuna_experiment()


[I 2026-06-07 09:21:31,445] A new study created in memory with name: no-name-efe40b8e-d25d-4edd-ae9d-b74e62e96aa8
[I 2026-06-07 09:22:52,083] Trial 0 finished with value: 0.6202709836354038 and parameters: {'C': 0.0017256468746803082, 'kernel': 'rbf'}. Best is trial 0 with value: 0.6202709836354038.
[I 2026-06-07 09:24:36,642] Trial 1 finished with value: 0.893894070033433 and parameters: {'C': 6.2807106852925845, 'kernel': 'rbf'}. Best is trial 1 with value: 0.893894070033433.
[I 2026-06-07 09:26:37,778] Trial 2 finished with value: 0.8933661798345944 and parameters: {'C': 10.493119101381234, 'kernel': 'rbf'}. Best is trial 1 with value: 0.893894070033433.
[I 2026-06-07 09:27:52,602] Trial 3 finished with value: 0.628893190216435 and parameters: {'C': 0.00995728317258033, 'kernel': 'rbf'}. Best is trial 1 with value: 0.893894070033433.
[I 2026-06-07 09:28:40,416] Trial 4 finished with value: 0.8486714763329227 and parameters: {'C': 0.04522815549915123, 'kernel': 'linear'}. Best is tri

# Random Forest Block

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import RandomForestClassifier

# Step 1: Remap the class labels from [-1, 0, 1] to
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# --- SPLIT ORIGINAL RAW TEXT DATA FIRST (Prevents Leakage) ---
X_raw = df['clean_comment']
y_raw = df['category']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)


# Function to log results in MLflow
def log_mlflow(model_name, model, vectorizer, X_train_raw, X_test_raw, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_ClassWeights_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        # 1. Fit/Transform vectorizer ONLY on training set text
        X_train_vec = vectorizer.fit_transform(X_train_raw)
        X_test_vec = vectorizer.transform(X_test_raw)

        # 2. Compute class balance weights ONLY for the training records
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

        # 3. Train model using sample weights
        model.fit(X_train_vec, y_train, sample_weight=sample_weights)
        y_pred = model.predict(X_test_vec)

        # Log metrics
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log nested classification report metrics
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log both artifacts
        mlflow.sklearn.log_model(model, f"{model_name}_model")
        mlflow.sklearn.log_model(vectorizer, f"{model_name}_vectorizer")


# Step 6: Optuna objective function with Class Weights
def objective_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 5, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)

    # 1. Isolate TF-IDF fitting to training split
    vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=1000)
    X_train_vec = vectorizer.fit_transform(X_train_raw)
    X_test_vec = vectorizer.transform(X_test_raw)

    # 2. Compute dynamic weights inside optimization step
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    # 3. Initialize and evaluate model
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )

    model.fit(X_train_vec, y_train, sample_weight=sample_weights)
    y_pred = model.predict(X_test_vec)
    return accuracy_score(y_test, y_pred)


# Step 7: Run Optuna for Random Forest, log the best model setup
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_rf, n_trials=30)

    best_params = study.best_params
    best_model = RandomForestClassifier(
        n_estimators=best_params['n_estimators'],
        max_depth=best_params['max_depth'],
        min_samples_split=best_params['min_samples_split'],
        random_state=42
    )

    final_vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=1000)

    # Log the baseline objects cleanly
    log_mlflow("RandomForest", best_model, final_vectorizer, X_train_raw, X_test_raw, y_train, y_test)

# Run the experiment for Random Forest
run_optuna_experiment()


[I 2026-06-07 07:04:14,606] A new study created in memory with name: no-name-97b9b546-7b44-454d-9e97-d1762ee5aa05
[I 2026-06-07 07:04:23,506] Trial 0 finished with value: 0.6902588936487242 and parameters: {'n_estimators': 177, 'max_depth': 18, 'min_samples_split': 10}. Best is trial 0 with value: 0.6902588936487242.
[I 2026-06-07 07:04:29,858] Trial 1 finished with value: 0.6766623207301173 and parameters: {'n_estimators': 116, 'max_depth': 12, 'min_samples_split': 3}. Best is trial 0 with value: 0.6902588936487242.
[I 2026-06-07 07:04:35,728] Trial 2 finished with value: 0.6429502700689141 and parameters: {'n_estimators': 260, 'max_depth': 6, 'min_samples_split': 6}. Best is trial 0 with value: 0.6902588936487242.
[I 2026-06-07 07:04:45,095] Trial 3 finished with value: 0.6906314024958092 and parameters: {'n_estimators': 99, 'max_depth': 20, 'min_samples_split': 8}. Best is trial 3 with value: 0.6906314024958092.
[I 2026-06-07 07:04:52,448] Trial 4 finished with value: 0.656546842987

# Naive Bayes

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.naive_bayes import MultinomialNB

# Step 1: Remap the class labels from [-1, 0, 1] to
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# --- SPLIT ORIGINAL RAW TEXT DATA FIRST (Prevents Leakage) ---
X_raw = df['clean_comment']
y_raw = df['category']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)


# Function to log results in MLflow
def log_mlflow(model_name, model, vectorizer, X_train_raw, X_test_raw, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_ClassWeights_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        # 1. Fit/Transform vectorizer ONLY on training set text
        X_train_vec = vectorizer.fit_transform(X_train_raw)
        X_test_vec = vectorizer.transform(X_test_raw)

        # 2. Compute class balance weights ONLY for the training records
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

        # 3. Train model using sample weights
        model.fit(X_train_vec, y_train, sample_weight=sample_weights)
        y_pred = model.predict(X_test_vec)

        # Log metrics
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log nested classification report metrics
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log both artifacts
        mlflow.sklearn.log_model(model, f"{model_name}_model")
        mlflow.sklearn.log_model(vectorizer, f"{model_name}_vectorizer")


# Step 6: Optuna objective function with Class Weights
def objective_nb(trial):
    alpha = trial.suggest_float('alpha', 1e-3, 1e1, log=True)
    fit_prior = trial.suggest_categorical('fit_prior', [True, False])

    # 1. Isolate TF-IDF fitting to training split
    vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=1000)
    X_train_vec = vectorizer.fit_transform(X_train_raw)
    X_test_vec = vectorizer.transform(X_test_raw)

    # 2. Compute dynamic weights inside optimization step
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    # 3. Initialize and evaluate model
    model = MultinomialNB(
        alpha=alpha,
        fit_prior=fit_prior
    )

    model.fit(X_train_vec, y_train, sample_weight=sample_weights)
    y_pred = model.predict(X_test_vec)
    return accuracy_score(y_test, y_pred)


# Step 7: Run Optuna for Naive Bayes, log the best model setup
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_nb, n_trials=30)

    best_params = study.best_params
    best_model = MultinomialNB(
        alpha=best_params['alpha'],
        fit_prior=best_params['fit_prior']
    )

    final_vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=1000)

    # Log the baseline objects cleanly
    log_mlflow("NaiveBayes", best_model, final_vectorizer, X_train_raw, X_test_raw, y_train, y_test)

# Run the experiment for Naive Bayes
run_optuna_experiment()


[I 2026-06-07 07:11:12,083] A new study created in memory with name: no-name-6231e1c5-2717-43cc-9e73-ec8767f886fe
[I 2026-06-07 07:11:14,712] Trial 0 finished with value: 0.8442405216131369 and parameters: {'alpha': 0.4029272343082738, 'fit_prior': False}. Best is trial 0 with value: 0.8442405216131369.
[I 2026-06-07 07:11:18,566] Trial 1 finished with value: 0.8471383723738227 and parameters: {'alpha': 0.046238936881222534, 'fit_prior': False}. Best is trial 1 with value: 0.8471383723738227.
[I 2026-06-07 07:11:21,033] Trial 2 finished with value: 0.8483458101907752 and parameters: {'alpha': 0.002962217141970138, 'fit_prior': False}. Best is trial 2 with value: 0.8483458101907752.
[I 2026-06-07 07:11:24,106] Trial 3 finished with value: 0.8483458101907752 and parameters: {'alpha': 0.0027297582504736437, 'fit_prior': True}. Best is trial 2 with value: 0.8483458101907752.
[I 2026-06-07 07:11:26,551] Trial 4 finished with value: 0.8220236657812122 and parameters: {'alpha': 3.896656752407

# KNN

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import optuna
import inspect # Import inspect module
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.neighbors import KNeighborsClassifier

# Step 1: Remap the class labels from [-1, 0, 1] to
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# --- SPLIT ORIGINAL RAW TEXT DATA FIRST (Prevents Leakage) ---
X_raw = df['clean_comment']
y_raw = df['category']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)


# Function to log results in MLflow
def log_mlflow(model_name, model, vectorizer, X_train_raw, X_test_raw, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_ClassWeights_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        # 1. Fit/Transform vectorizer ONLY on training set text
        X_train_vec = vectorizer.fit_transform(X_train_raw)
        X_test_vec = vectorizer.transform(X_test_raw)

        # 2. Compute class balance weights ONLY for the training records
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

        # 3. Train model using sample weights conditionally
        fit_params = inspect.signature(model.fit).parameters
        if 'sample_weight' in fit_params:
            model.fit(X_train_vec, y_train, sample_weight=sample_weights)
        else:
            model.fit(X_train_vec, y_train)

        y_pred = model.predict(X_test_vec)

        # Log metrics
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log nested classification report metrics
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log both artifacts
        mlflow.sklearn.log_model(model, f"{model_name}_model")
        mlflow.sklearn.log_model(vectorizer, f"{model_name}_vectorizer")


# Step 6: Optuna objective function with Class Weights
def objective_knn(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 3, 25)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    metric = trial.suggest_categorical('metric', ['euclidean', 'cosine'])

    # 1. Isolate TF-IDF fitting to training split
    vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=1000)
    X_train_vec = vectorizer.fit_transform(X_train_raw)
    X_test_vec = vectorizer.transform(X_test_raw)

    # 2. Compute dynamic weights inside optimization step (not used for KNN)
    # sample_weights = compute_sample_weight(class_weight='balanced', y=y_train) # Removed since KNN does not support it

    # 3. Initialize and evaluate model
    model = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights=weights,
        metric=metric
    )

    # Remove sample_weight from here as KNeighborsClassifier does not support it
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    return accuracy_score(y_test, y_pred)


# Step 7: Run Optuna for KNN, log the best model setup
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_knn, n_trials=30)

    best_params = study.best_params
    best_model = KNeighborsClassifier(
        n_neighbors=best_params['n_neighbors'],
        weights=best_params['weights'],
        metric=best_params['metric']
    )

    final_vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=1000)

    # Log the baseline objects cleanly
    log_mlflow("KNN", best_model, final_vectorizer, X_train_raw, X_test_raw, y_train, y_test)

# Run the experiment for KNN
run_optuna_experiment()

[I 2026-06-07 09:15:33,251] A new study created in memory with name: no-name-ffc97843-1591-4cbb-b28d-ed7b4a4c72d1
[I 2026-06-07 09:15:42,980] Trial 0 finished with value: 0.5490249556798036 and parameters: {'n_neighbors': 23, 'weights': 'distance', 'metric': 'cosine'}. Best is trial 0 with value: 0.5490249556798036.
[I 2026-06-07 09:15:53,710] Trial 1 finished with value: 0.5378426292104187 and parameters: {'n_neighbors': 20, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 0 with value: 0.5490249556798036.
[I 2026-06-07 09:16:04,282] Trial 2 finished with value: 0.4753852447838538 and parameters: {'n_neighbors': 7, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 0 with value: 0.5490249556798036.
[I 2026-06-07 09:16:15,120] Trial 3 finished with value: 0.4125187508523115 and parameters: {'n_neighbors': 20, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 0 with value: 0.5490249556798036.
[I 2026-06-07 09:16:25,447] Trial 4 finished with value: 0.541524614755

# Logistic Regression

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression

# Step 1: Remap the class labels from [-1, 0, 1] to
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# --- SPLIT ORIGINAL RAW TEXT DATA FIRST (Prevents Leakage) ---
X_raw = df['clean_comment']
y_raw = df['category']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)


# Function to log results in MLflow
def log_mlflow(model_name, model, vectorizer, X_train_raw, X_test_raw, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_ClassWeights_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        # 1. Fit/Transform vectorizer ONLY on training set text
        X_train_vec = vectorizer.fit_transform(X_train_raw)
        X_test_vec = vectorizer.transform(X_test_raw)

        # 2. Compute class balance weights ONLY for the training records
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

        # 3. Train model using sample weights
        model.fit(X_train_vec, y_train, sample_weight=sample_weights)
        y_pred = model.predict(X_test_vec)

        # Log metrics
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log nested classification report metrics
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log both artifacts
        mlflow.sklearn.log_model(model, f"{model_name}_model")
        mlflow.sklearn.log_model(vectorizer, f"{model_name}_vectorizer")


# Step 6: Optuna objective function with Class Weights
def objective_lr(trial):
    C = trial.suggest_float('C', 1e-4, 1e2, log=True)
    solver = trial.suggest_categorical('solver', ['lbfgs', 'saga'])

    # 1. Isolate TF-IDF fitting to training split
    vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=1000)
    X_train_vec = vectorizer.fit_transform(X_train_raw)
    X_test_vec = vectorizer.transform(X_test_raw)

    # 2. Compute dynamic weights inside optimization step
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    # 3. Initialize and evaluate model
    model = LogisticRegression(
        C=C,
        solver=solver,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train_vec, y_train, sample_weight=sample_weights)
    y_pred = model.predict(X_test_vec)
    return accuracy_score(y_test, y_pred)


# Step 7: Run Optuna for Logistic Regression, log the best model setup
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lr, n_trials=30)

    best_params = study.best_params
    best_model = LogisticRegression(
        C=best_params['C'],
        solver=best_params['solver'],
        max_iter=1000,
        random_state=42
    )

    final_vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=1000)

    # Log the baseline objects cleanly
    log_mlflow("LogisticRegression", best_model, final_vectorizer, X_train_raw, X_test_raw, y_train, y_test)

# Run the experiment for Logistic Regression
run_optuna_experiment()


[I 2026-06-07 07:12:57,295] A new study created in memory with name: no-name-5063e315-77e4-4546-b88c-317e19457f22
[I 2026-06-07 07:12:59,977] Trial 0 finished with value: 0.7821782178217822 and parameters: {'C': 0.003262497679311362, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.7821782178217822.
[I 2026-06-07 07:13:02,759] Trial 1 finished with value: 0.90050712388312 and parameters: {'C': 2.500544644403702, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.90050712388312.
[I 2026-06-07 07:13:06,056] Trial 2 finished with value: 0.8270949046124124 and parameters: {'C': 0.047828674387506466, 'solver': 'saga'}. Best is trial 1 with value: 0.90050712388312.
[I 2026-06-07 07:13:09,821] Trial 3 finished with value: 0.8995411736295581 and parameters: {'C': 65.17935343482434, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.90050712388312.
[I 2026-06-07 07:13:12,552] Trial 4 finished with value: 0.7848345810190775 and parameters: {'C': 0.004731415975933482, 'solver': 'saga'}. Best is tr